# Phase 2/3: Validation/Application on Non-Invasive (EGI) Data

## 1. Preprocessing on EGI Data

This section converts raw EGI recordings (.mff) into preprocessed .fif files
suitable for zEEG feature extraction and ED rate computation.

### 1.1 MFF → Preprocessed FIF

Reads a raw EGI recording (.mff), picks the zEEG channels, resamples,
band-pass filters, crops to the first NREM period, applies SMA detrending
and a notch filter, then saves a single preprocessed .fif file.

In [ ]:
import mne
import os
import numpy as np
import pandas as pd
import joblib
from IPython.display import clear_output
from zeeg_utils import raw_chan_to_feat

target_sample_rate = 1000     # Hz
eeg_bp_freq = [0.1, 70]
notch_freqs = (50, 100, 150, 200)
sma_window_sec = 2  # SMA detrending window in seconds

zeeg1_id = 'E227'
zeeg2_id = 'E254'
selected_set = [zeeg1_id, zeeg2_id]

In [ ]:
mff_file = 'recording.mff'
output_folder = 'fif_zeeg'

TMIN_MIN = 8      # start of first NREM (minutes from recording start)
TMAX_MIN = 65   # end of first NREM (minutes from recording start)

# Read MFF and pick zEEG channels
egi_full_montage = mne.channels.make_standard_montage('GSN-HydroCel-257')
raw = mne.io.read_raw_egi(mff_file)
raw.rename_channels({'VREF': 'Cz'})
raw.set_montage(egi_full_montage)
raw.pick_channels(selected_set)
raw.load_data()

# Resample and band-pass filter
raw.resample(target_sample_rate)
raw.filter(eeg_bp_freq[0], eeg_bp_freq[1])

# Crop to first NREM period
tmin = TMIN_MIN * 60
tmax = min(TMAX_MIN * 60, raw.times[-1])
raw.crop(tmin=tmin, tmax=tmax)

# SMA detrending
data = raw.get_data()
sma_window = int(raw.info['sfreq'] * sma_window_sec)
raw._data = np.array([
    ch - np.convolve(ch, np.ones(sma_window) / sma_window, mode='same')
    for ch in data
])

# Notch filter
raw.notch_filter(notch_freqs)

# Save
os.makedirs(output_folder, exist_ok=True)
base_name = os.path.splitext(os.path.basename(mff_file))[0]
out_path = os.path.join(output_folder, f'{base_name}.fif')
raw.save(out_path, overwrite=True)
print(f'Saved: {out_path}')

### 1.2 Data Cleaning

Visually inspect the preprocessed data, mark bad segments as annotations,
then save the cleaned file.

In [ ]:
fif_path = os.path.join(output_folder, f'{base_name}.fif')
raw = mne.io.read_raw_fif(fif_path, preload=True)
raw.plot(duration=60*5, block=True)

# After marking bad segments interactively, save the cleaned file (with annotations)
raw.save(fif_path, overwrite=True)
print(f'Saved cleaned: {fif_path}')

### 1.3 Batch Feature Extraction

Computes zEEG features for every subject in the folder and saves
one .pkl file per subject. These files are later read by
the ED rate extraction function.

In [ ]:
processed_folder = 'fif_zeeg'

fif_files = [f for f in os.listdir(processed_folder) if f.endswith('.fif')]
subjects = [os.path.splitext(f)[0] for f in fif_files]

for subj in subjects:
    print(f'Processing subject: {subj}')

    fif_path = os.path.join(processed_folder, f'{subj}.fif')
    subject_data = {}

    raw = mne.io.read_raw_fif(fif_path, preload=True)

    for ch in selected_set:
        try:
            subject_data[ch] = raw_chan_to_feat(raw, ch, subj, depth=False)
            clear_output()
        except Exception as e:
            print(f'Error processing {ch} for {subj}: {e}')
            continue

    output_path = os.path.join(processed_folder, f'{subj}_processed.pkl')
    joblib.dump(subject_data, output_path)
    print(f'Saved: {output_path}')

print('Processing complete!')

## 2. ED Rate Extraction

Computes the ED rate (EDs per minute) for each subject from
pre-computed zEEG features using the trained model.

In [ ]:
def extract_rate_per_bin(model, channel_1, channel_2, threshold, bin_length, sec_limit):
    """
    Computes ED rate (EDs per minute) for HC and EPI groups from feature data.

    This function iterates through subjects, bins their data, runs the model,
    applies a refractory period, and calculates the rate for each bin.

    Args:
        model: The trained XGBoost model object.
        channel_1 (str): Name of the first zEEG channel key (e.g., 'E227').
        channel_2 (str): Name of the second zEEG channel key (e.g., 'E254').
        threshold (float): Probability threshold for ED detection (0 to 1).
        bin_length (int): Duration of each analysis bin in minutes.
        sec_limit (int): Refractory period in seconds after a detection.

    Returns:
        pd.DataFrame: A DataFrame with ["subject", "EDs_per_min", "group"]
                      for each bin of each subject.
    """

    sampling_hz = 4
    samples_per_min = sampling_hz * 60  # 240
    bin_stride = bin_length * samples_per_min
    refractory = int(sec_limit * sampling_hz)

    rows = []

    def process_group(group_label, subjects, store):
        for subj in subjects:
            zeeg1 = store[subj][channel_1]
            zeeg2 = store[subj][channel_2]
            curr_feat = pd.concat([zeeg1, zeeg2], axis=1, ignore_index=True)
            curr_feat.columns = [f"zeeg1_{c}" for c in zeeg1.columns] + [f"zeeg2_{c}" for c in zeeg2.columns]

            feat_names = model.get_booster().feature_names
            X_all = curr_feat[feat_names]

            part = 0
            n = len(X_all)
            for start in range(0, n, bin_stride):
                stop = start + bin_stride
                X = X_all.iloc[start:stop, :]

                if len(X) < (bin_stride // 2):
                    continue

                proba = model.predict_proba(X)[:, 1]

                result = np.zeros_like(proba, dtype=int)
                i = 0
                while i < len(proba):
                    if proba[i] >= threshold:
                        result[i] = 1
                        i += refractory
                    else:
                        i += 1

                rate = result.sum() / (len(result) / samples_per_min)

                rows.append({
                    "subject": f"{subj}_{part}",
                    "EDs_per_min": rate,
                    "group": group_label
                })

                part += 1

    process_group("HC", subjects_HC, hc)
    process_group("EPI", subjects_EPI, epi)

    df = pd.DataFrame(rows, columns=["subject", "EDs_per_min", "group"])
    return df

In [ ]:
subjects_HC = ['HC1', 'HC2', 'HC3']  # example healthy control ids
subjects_EPI = ['EPI1', 'EPI2', 'EPI3']  # example epilepsy patient ids
# load pre-computed features (output of batch feature extraction)
hc = joblib.load('v1_HC.pkl')
epi = joblib.load('v1_EPI.pkl')

In [ ]:
threshold = 0.7    # probability threshold for ED detection
model_im = joblib.load('zeeg_model.pkl')
bin_length = 20 # separate into 20 min bins
sec_limit = 2   # refractory period in seconds

df = extract_rate_per_bin(model_im, zeeg1_id, zeeg2_id, threshold, bin_length, sec_limit)
df

In [ ]:
# Save results
df.to_csv('zeeg_eds_per_min.csv', index=False)